In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 42
TARGET = "Churned"
DATA_PATH = Path("ecommerce_customer_churn_dataset.csv")

print("Environment:", sys.executable)
print("Dataset found:", DATA_PATH.exists())

Environment: c:\Users\User\anaconda3\python.exe
Dataset found: True


In [2]:
from sklearn.model_selection import train_test_split, StratifiedKFold

df = pd.read_csv(DATA_PATH)

X = df.drop(columns=TARGET)
y = df[TARGET]

Xtr, Xte, ytr, yte = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

cv5 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Dataset shape:", df.shape)
print("Training features:", Xtr.shape)
print("Test features:", Xte.shape)
print(f"Training churn rate: {ytr.mean():.1%}")
print(f"Test churn rate: {yte.mean():.1%}")
print("Overlapping row indices:", len(Xtr.index.intersection(Xte.index)))

Dataset shape: (50000, 25)
Training features: (40000, 24)
Test features: (10000, 24)
Training churn rate: 28.9%
Test churn rate: 28.9%
Overlapping row indices: 0


In [3]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CAT = ["Gender", "Country", "Signup_Quarter"]

class Cleaner(BaseEstimator, TransformerMixin):
    """Rule-based correction of invalid values (4.1), dropping City (4.3), engineered features (4.5)."""
    def __init__(self, fe=True, drop_city=True, clean=True, logs=False):
        self.fe, self.drop_city, self.clean, self.logs = fe, drop_city, clean, logs
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        if self.clean:
            X.loc[(X.Age < 18) | (X.Age > 100), "Age"] = np.nan          # impossible ages -> missing
            X.loc[X.Total_Purchases < 0, "Total_Purchases"] = np.nan      # negative counts -> missing
            X["Total_Purchases"] = X.Total_Purchases.round()              # a count must be whole
            for c in ["Cart_Abandonment_Rate", "Discount_Usage_Rate"]:    # percentages capped at 100
                X[c] = X[c].clip(upper=100)
        if self.drop_city and "City" in X:
            X = X.drop(columns="City")
        if self.fe:                                                       # Table 4.2
            m = X.Membership_Years.clip(lower=0.1)
            X["Purchase_Frequency"]    = X.Total_Purchases / m
            X["Estimated_Spend"]       = X.Total_Purchases * X.Average_Order_Value
            X["Engagement_Index"]      = X.Login_Frequency * X.Session_Duration_Avg * X.Pages_Per_Session
            X["Service_Calls_per_Year"] = X.Customer_Service_Calls / m
            X["Recency_to_Tenure"]     = X.Days_Since_Last_Purchase / (m * 365)
        if self.logs:                                                     # only used in the rejected log-transform test
            cols = ["Average_Order_Value", "Total_Purchases", "Returns_Rate", "Days_Since_Last_Purchase",
                    "Lifetime_Value", "Membership_Years", "Wishlist_Items", "Product_Reviews_Written"]
            if self.fe:
                cols += ["Purchase_Frequency", "Estimated_Spend", "Engagement_Index", "Service_Calls_per_Year"]
            for c in cols:
                X[c] = np.log1p(X[c].clip(lower=0))
        return X

class Winsor(BaseEstimator, TransformerMixin):
    """Cap each numeric column at its 1st and 99th percentile (limits learned from training data)."""
    def __init__(self, lo=0.01, hi=0.99):
        self.lo, self.hi = lo, hi
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.cols_ = list(X.columns); self.l_ = X.quantile(self.lo); self.h_ = X.quantile(self.hi)
        return self
    def get_feature_names_out(self, names=None):
        return np.array(self.cols_)
    def transform(self, X):
        return pd.DataFrame(X).clip(self.l_, self.h_, axis=1).values

def make_pre(fe=True, winsor=True, scale=True, clean=True, logs=False):
    num = Pipeline(([("w", Winsor())] if winsor else [])
                   + [("imp", SimpleImputer(strategy="median"))]
                   + ([("sc", StandardScaler())] if scale else []))
    ct = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), CAT),
         ("num", num, make_column_selector(dtype_include=np.number))],
        verbose_feature_names_out=False)
    ct.set_output(transform="pandas")
    return Pipeline([("clean", Cleaner(fe=fe, clean=clean, logs=logs)), ("ct", ct)])


In [4]:
preparation_check = make_pre()
Xtr_check = preparation_check.fit_transform(Xtr, ytr)

print("Prepared training shape:", Xtr_check.shape)
print("Missing values remaining:", Xtr_check.isna().sum().sum())
print("All values finite:", np.isfinite(Xtr_check.to_numpy()).all())

Prepared training shape: (40000, 37)
Missing values remaining: 0
All values finite: True


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

def model_pipeline(model):
    return Pipeline([
        ("preparation", make_pre()),
        ("model", model)
    ])

base_models = {
    "Logistic Regression": model_pipeline(
        LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    ),
    "Random Forest": model_pipeline(
        RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    ),
    "Histogram Gradient Boosting": model_pipeline(
        HistGradientBoostingClassifier(
            class_weight="balanced",
            early_stopping=False,
            random_state=RANDOM_STATE
        )
    )
}

for name in base_models:
    print(name, "— ready")

Logistic Regression — ready
Random Forest — ready
Histogram Gradient Boosting — ready


In [ ]:
from sklearn.model_selection import cross_validate

scoring = {
    "roc_auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

baseline_rows = []
baseline_fold_results = {}

for name, pipeline in base_models.items():
    print(f"Running {name}...", flush=True)

    scores = cross_validate(
        pipeline,
        Xtr,
        ytr,
        cv=cv5,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
        error_score="raise"
    )

    baseline_fold_results[name] = scores

    baseline_rows.append({
        "Model": name,
        "Train ROC-AUC": scores["train_roc_auc"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean(),
        "CV ROC-AUC SD": scores["test_roc_auc"].std(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1": scores["test_f1"].mean(),
        "Mean fit time (s)": scores["fit_time"].mean()
    })

    print(f"Completed {name}", flush=True)

baseline_results = pd.DataFrame(baseline_rows).set_index("Model")
display(baseline_results.round(4))

Running Logistic Regression...
Completed Logistic Regression
Running Random Forest...
Completed Random Forest
Running Histogram Gradient Boosting...
Completed Histogram Gradient Boosting


,Train ROC-AUC,CV ROC-AUC,CV ROC-AUC SD,CV Precision,CV Recall,CV F1,Mean fit time (s)
Model,,,,,,,
Logistic Regression,0.7997,0.7983,0.0076,0.5159,0.7326,0.6054,0.2110
Random Forest,1.0000,0.9213,0.0043,0.9214,0.7490,0.8263,1.4732
Histogram Gradient Boosting,0.9709,0.9259,0.0040,0.8367,0.8455,0.8410,1.0437
